In [7]:
from pathlib import Path

import pandas as pd
from seaborn import displot
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

load_dotenv(dotenv_path=Path('etl/.env'))

# 1. Conexión a la Base de Datos
load_dotenv()
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db = os.getenv('DB_NAME')

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{db}')

# Extraemos la tabla específica para Apriori
query = "SELECT tienda, familia_producto, tipo_promocion, quiebre_stock FROM dw_promociones_apriori"
df = pd.read_sql(query, engine)

In [8]:
print("============================================================")
print("ALGORITMO APRIORI: REGLAS DE ASOCIACIÓN (MERCADOS LA COLINA)")
print("============================================================")

# 2. Preprocesamiento (Transformar a formato transaccional)
# Convertimos el 1 y 0 a texto para que las reglas sean fáciles de leer
df['quiebre_stock'] = df['quiebre_stock'].apply(lambda x: 'Quiebre_SI' if x == 1 else 'Quiebre_NO')

# Aplicamos One-Hot Encoding y convertimos a booleanos (True/False) como exige el algoritmo
df_encoded = pd.get_dummies(df).astype(bool)

# 3. Aplicar el Algoritmo Apriori
# Buscamos combinaciones que aparezcan en al menos el 2% de los datos (min_support=0.02)
frequent_itemsets = apriori(df_encoded, min_support=0.02, use_colnames=True)

# 4. Generar las Reglas de Asociación
# Queremos reglas que tengan al menos un 50% de confianza
reglas = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# 5. Filtrar: SOLO nos interesan las reglas que terminan en "Quiebre_SI"
reglas_quiebre = reglas[reglas['consequents'].apply(lambda x: 'quiebre_stock_Quiebre_SI' in x)].copy()

# Limpiar los nombres para que se vean bonitos en la tabla
reglas_quiebre['antecedents'] = reglas_quiebre['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas_quiebre['consequents'] = 'Quiebre_SI'

# Ordenar por Confianza (Confidence) y Lift
reglas_quiebre = reglas_quiebre.sort_values(by=['confidence', 'lift'], ascending=[False, False])

# Mostrar el TOP 10 de reglas más peligrosas
columnas_mostrar = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
df_final = reglas_quiebre[columnas_mostrar].head(10)

ALGORITMO APRIORI: REGLAS DE ASOCIACIÓN (MERCADOS LA COLINA)


In [9]:
# Crear una copia para no alterar los valores numéricos originales
df_display = df_final.copy()

df_display['support'] = df_display['support'].map('{:.2%}'.format)
df_display['confidence'] = df_display['confidence'].map('{:.2%}'.format)
df_display['lift'] = df_display['lift'].map('{:.2f}'.format)

display(df_display)  # o print(df_display)

,antecedents,consequents,support,confidence,lift
28,"tienda_Jinotepe, tipo_promocion_2x1",Quiebre_SI,2.62%,53.35%,1.52
36,"tipo_promocion_2x1, tienda_Managua Sur",Quiebre_SI,2.55%,52.84%,1.50
